# EPIC Quickstart — Your First Contest, Step by Step

Welcome to **EPIC** (ELIOS Predictive Intelligence Challenge). Unlike a classic Kaggle-style competition, EPIC does not hand you a dataset: a **digital twin** — a simulated physical system such as a pump, a motor, or a smart building — runs *live* on the server, and you collect your own training data by listening to its sensor stream in real time. Collecting good data is part of the challenge, exactly as it would be for an engineer instrumenting a real machine.

Every contest follows the same **two-phase structure**, designed so that nobody can cheat by submitting "predictions" of data they have already seen:

1. **Observation phase** — the simulation runs and streams sensor readings to registered participants over a WebSocket. This is your data-collection and model-training window.
2. **Evaluation phase** — the simulation keeps running, but the stream is closed. The server privately records the *ground truth* for this window; nobody can see it.
3. **Submission window** — once the evaluation phase ends, you submit a forecast covering every step of the evaluation window. It is scored automatically against the recorded ground truth, and the leaderboard updates.

In this notebook you will log in, find a contest, register, collect live data, build the simplest possible forecast (a *persistence baseline*), submit it, and read your score. Every step explains not just *what* to run but *why*.

## 1. Installation

The cell below installs the official participant SDK, `epic-elios-client`, together with the optional notebook extras (pandas and matplotlib, used here for tables and plots). The SDK wraps the platform's REST API and WebSocket stream so you never have to handle HTTP or reconnection logic yourself. Run it once per environment.

In [ ]:
%pip install "epic-elios-client[notebook]"

## 2. Configure the Client and Log In

Two things to know about accounts before this cell:

- **You cannot self-register as a participant.** Accounts are created either by an instructor/administrator, or — most commonly — through a personal **invitation link** the contest organizer sent to your email. If you followed such a link and completed the registration form, the username is your email address and the password is the one you chose on that form.
- Logging in returns a **JWT token** that the client stores and attaches to every subsequent call. Tokens expire after about an hour; if calls suddenly start failing with authentication errors, simply run this cell again.

Replace the credentials below with your own.

In [ ]:
from epic_client import EPICClient

SERVER_URL = "https://epic.elioslab.net"
client = EPICClient(SERVER_URL)

client.login("your-username", "your-password")

## 3. Browse the Contests You Can See

The listing below is **personalized** — the platform only shows you contests you are allowed to access. Contests have one of three visibility levels:

- **PUBLIC** — listed for every logged-in user, and anyone may register.
- **INVITATION_ONLY** — joining requires an invitation from the organizer. You will see such a contest in your listing *only if* your email has been invited (or you are already registered).
- **PRIVATE** — like invitation-only, but additionally hidden from everyone who isn't invited, registered, or the organizer.

So if a contest your professor mentioned doesn't appear here, the most likely explanation is that the invitation went to a different email address than the one on your account — ask the organizer to invite the right one.

Two columns of the table deserve special attention, because together they define *exactly what you must submit*:

- **`eval_steps`** — the number of values you must predict per variable. It equals `prediction_horizon_seconds × sampling_rate_hz`: a 60-second horizon sampled at 10 Hz means 600 predicted values.
- **`target_variables`** — the subset of sensors that will actually be **scored**. A contest may stream five sensors but evaluate your forecast on only two of them. You still receive all sensors during the observation phase — the non-target ones are free extra features for your model.

In [ ]:
import pandas as pd

contests = client.list_contests(status="ACTIVE")

rows = []
for c in contests:
    task_cfg = c.get("tasks", [{}])[0].get("configuration", {})
    rows.append({
        "contest_id":                c.get("contest_id"),
        "name":                      c.get("name"),
        "visibility":                c.get("visibility"),
        "twin_id":                   c.get("twin_id"),
        "sampling_rate_hz":          c.get("sampling_rate_hz"),
        "end_of_observation":        c.get("end_of_observation"),
        "prediction_horizon_seconds": c.get("prediction_horizon_seconds"),
        "eval_steps":                task_cfg.get("eval_steps"),
        "target_variables":          ", ".join(task_cfg.get("target_variables", [])),
    })

pd.DataFrame(rows)

## 4. Pick a Contest and Read Its Task Configuration

Copy a `contest_id` from the table above and paste it below. The cell then extracts the three numbers that drive everything downstream: how many steps to predict (`EVAL_STEPS`), which variables will be scored (`TARGET_VARIABLES`), and the sampling rate that links steps to seconds. Getting these from the contest itself — rather than hard-coding them — means this notebook works unchanged on any forecasting contest.

In [ ]:
CONTEST_ID = "replace-with-contest-id"  # copy a contest_id from the table above

# Read the task configuration from the platform via the SDK helper.
contest = client.get_contest(CONTEST_ID)
task_spec = client.get_task_spec(CONTEST_ID)
EVAL_STEPS = task_spec["eval_steps"]
TARGET_VARIABLES = task_spec.get("target_variables") or []
SAMPLING_RATE_HZ = task_spec["sampling_rate_hz"]

print(f"visibility       : {contest['visibility']}")
print(f"eval_steps       : {EVAL_STEPS}")
print(f"target_variables : {TARGET_VARIABLES}")
print(f"sampling_rate_hz : {SAMPLING_RATE_HZ} Hz")
print(f"evaluation window: {EVAL_STEPS / SAMPLING_RATE_HZ:.1f} s")

## 5. Register for the Contest

Registration links your account to this specific contest; without it you can neither receive the data stream nor submit. The rules follow the visibility level: for a **PUBLIC** contest this call always succeeds, while for a **PRIVATE** or **INVITATION_ONLY** contest the platform checks that the organizer invited *your* email address and rejects the registration otherwise — so being able to *see* a restricted contest (because you were invited) is exactly what entitles you to join it.

If you created your account by following an invitation link, you were **registered automatically** for that contest — in that case this call reports that you are already registered, which the SDK treats as success.

In [ ]:
client.register(CONTEST_ID)

## 6. Collect Live Observations — the Observation Phase

`collect()` opens the WebSocket stream and gathers observations for the requested duration, optionally mirroring them to a CSV file with one column per sensor so your dataset survives a kernel restart. Each observation carries a `sequence_id` (a gapless counter on the server side — if *you* see a gap, your connection dropped and those readings are gone forever, because the server never replays), a timestamp, and one numeric reading per configured sensor.

Three practical notes. First, you can only collect **while the observation phase is open** — after `end_of_observation` the stream rejects connections, and joining late means you simply have less data than participants who started earlier, just like in real monitoring. Second, choose `duration_seconds` deliberately: more data generally helps, but leave yourself time before the phase ends. Third, what you receive are **sensor readings, not ground truth** — the contest may have configured noise, drift, or outliers on each sensor, and part of the challenge is dealing with that.

In [ ]:
observations = await client.collect(
    CONTEST_ID,
    duration_seconds=30,          # adjust to match your contest's observation window
    csv_path="epic_observations.csv",
)
print(f"Collected {len(observations)} observations")
observations[:2]

## 7. Build a DataFrame

Each observation nests its readings in a `sensors` dictionary; flattening one sensor per column gives a tidy table to work with. Note the columns: *all* configured sensors appear, not only the target variables — remember the extras are legitimate model inputs.

In [ ]:
rows = []
for obs in observations:
    rows.append({
        "sequence_id": obs["sequence_id"],
        "timestamp":   obs["timestamp"],
        **obs["sensors"],
    })

df = pd.DataFrame(rows)
df.head()

## 8. Plot a Sensor Channel

Always look at your data before modeling it. The plot below shows the first sensor channel against `sequence_id`; with a mechanical twin you should see an oscillation, with a thermal one a slow trend. Jagged spikes are usually configured sensor noise or outliers — information about how hard the organizer made this contest.

In [ ]:
import matplotlib.pyplot as plt

sensor_columns = [c for c in df.columns if c not in {"sequence_id", "timestamp"}]
sensor_name = sensor_columns[0]

df.plot(x="sequence_id", y=sensor_name, figsize=(10, 4), title=f"{sensor_name} over time")
plt.xlabel("Sequence ID")
plt.ylabel(sensor_name)
plt.tight_layout()
plt.show()

## 9. Build the Simplest Possible Forecast — a Persistence Baseline

A *persistence* forecast predicts that every future value equals the last observed one. It sounds trivial, and it is — but it is the standard baseline of time-series forecasting: any model worth keeping must beat it, and submitting it first verifies your entire pipeline end to end before you invest in modeling.

The payload rules matter here. You must include **every target variable** with **exactly `eval_steps` values each** — a missing target or a wrong length is rejected with an explanatory error. Forecasts for non-target sensors are allowed but simply ignored by the scorer, so the cell below predicts only what will be scored.

In [ ]:
forecast_targets = TARGET_VARIABLES or sensor_columns
missing_targets = [target for target in forecast_targets if target not in df.columns]
if missing_targets:
    raise ValueError(f"Target variables not present in collected data: {missing_targets}")

last_values = df[forecast_targets].iloc[-1]

# Persist the last observed value for every target variable and every step.
payload = {
    "forecast": {
        target: [float(last_values[target])] * EVAL_STEPS
        for target in forecast_targets
    }
}

print(f"Forecast shape: {len(forecast_targets)} target variable(s) × {EVAL_STEPS} steps")
payload

## 10. Submit — the Submission Window

Submissions are only accepted **after the evaluation phase has fully elapsed** — that is, after `end_of_observation + prediction_horizon_seconds`. This is the platform's anti-cheating guarantee: by the time anyone may submit, the ground truth already exists on the server and nobody could have seen it, so every accepted forecast is genuinely prospective. Submitting earlier returns a clear `409` error telling you when the window opens; just wait and re-run this cell.

The response shows your submission in `PENDING` status — scoring happens asynchronously, usually within a second or two.

In [ ]:
submission = client.submit(
    contest_id=CONTEST_ID,
    task_id="forecasting",
    payload=payload,
)
submission

## 11. Check Scores and the Leaderboard

A submission moves from `PENDING` to `EVALUATED` (one score per metric per target variable) or `FAILED` (with the reason recorded — typically a malformed payload). The default metric is **MAE**, mean absolute error in the sensor's own units, where *lower is better*. By default you are scored against the clean **ground truth** — the noiseless latent state of the twin — so sensor noise does not unfairly penalize a good model.

The leaderboard keeps each participant's **best** submission, so you can resubmit as often as you like: a worse attempt never hurts your ranking. If your submission shows `PENDING`, wait a moment and re-run.

In [ ]:
scores = client.get_scores(CONTEST_ID)
leaderboard = client.get_leaderboard(CONTEST_ID)

print("=== My Submissions ===")
for sub in scores.get("submissions", []):
    print(f"  {sub['submission_id'][:8]}…  status={sub['status']}  "
          f"scores={[round(s['value'], 4) for s in sub.get('scores', [])]}")

print("\n=== Leaderboard ===")
for entry in leaderboard.get("entries", []):
    print(f"  Rank {entry['rank']}  user={entry['user_id'][:8]}…  score={entry['score']:.4f}")

## Next Steps

The persistence baseline is a starting point, not a strategy. Natural improvements, roughly in order of effort: exploit the **physics** of the twin (the companion notebook `mass_spring_damper_forecasting.ipynb` estimates the actual equation of motion and integrates it forward — a beautiful example of a model that nearly matches the simulator); use classical time-series models (AR, ARIMA, exponential smoothing) on each target; or train a learned regressor using *all* streamed sensors as features for the target variables. Whatever you build, the loop stays the same: collect during observation, predict `eval_steps` values per target, submit when the window opens, and let the leaderboard keep your best.